# 13 — Matched LightGBM versus calibrated MLP + Extra Trees

Run the complete **25 matching outer folds** requested for comparison with the supplied LightGBM notebook. The recovered training pool contains **14,439 postings, 693 frauds, 3,555 groups and 331 fraud groups**. All 25 original outer-fold signatures are verified before fitting. Existing data and experiments are preserved.

The original training CSV bytes are unavailable: its file checksum differs from the reconstruction. A **fresh LightGBM replay** provides the paired comparator; the saved historical scores are audited separately. The split is recovered from saved job IDs rather than regenerated by a potentially different splitter.

**Completed results are included in `reports/13_reference_comparison/`. Run All displays them by default; no training starts.** To reproduce the comparison, install `requirements-13.txt` with Python 3.11.9, select that environment and set `RUN_TRAINING=True`. The launcher uses the notebook's interpreter. Fixed training inputs download automatically from the GitHub release; no other notebooks or Kaggle credentials are needed. See `RUN_NOTEBOOK_13.md`.

Both pipelines use A2 blocks. LightGBM retains its original 2,000-feature TF-IDF cap and [15,31] leaf grid; the previously selected ensemble retains 30,000 TF-IDF features, the original MLP, leaf-1 Extra Trees and fixed equal weights. This compares selected pipelines, not algorithms under identical feature dimensionality.

Three grouped inner folds choose LightGBM settings by mean AP and choose each pipeline’s threshold by maximum inner OOF F1. Ensemble components are sigmoid-calibrated using separate groups inside each fit. Outer labels are used only for assessment.

Primary report columns: **PR-AUC (computed as average precision/AP), fraud precision, recall and F1**. Means and SDs are across the same 25 outer folds. Precision/recall in the highest-scoring 5% are secondary review-budget measures. Do not compare a partial ensemble mean against all 25 historical LightGBM folds.

No reserved test predictions are computed. Some original test IDs may have entered earlier experiments using the different split; the project should not claim that those IDs form a completely untouched final test.

In [1]:
from pathlib import Path
import json, subprocess, sys
import pandas as pd
from IPython.display import display
ROOT = next(p for p in (Path.cwd(), *Path.cwd().parents) if (p / 'src/d1_reference_comparison.py').is_file())
if str(ROOT) not in sys.path:
    sys.path.insert(0, str(ROOT))
BASE = ROOT / 'data/processed/13_reference_comparison'
PUBLISHED = ROOT / 'reports/13_reference_comparison'
RUN_TRAINING = False  # True launches a fresh/resumable 25-fold run in the background.
print('Completed comparison: 25 matching outer folds per model.')


Completed comparison: 25 matching outer folds per model.


## Start a reproduction only when needed

The original results remain read-only under `reports/13_reference_comparison/`. A reproduction writes to a fingerprinted folder under ignored `data/processed/13_reference_comparison/`. Completed stages resume when data, source and package fingerprints match. Changes to setup/packaging create a different fingerprint from the original run.

The full run can take several hours. The launch cell finishes immediately because training uses a separate background process. The final live-monitor cell displays that worker's progress. Do not run multiple copies.

In [2]:
if RUN_TRAINING:
    import os
    env = dict(os.environ, REFERENCE13_PYTHON=sys.executable)
    subprocess.run([sys.executable, '-m', 'src.run_reference13'], cwd=ROOT, env=env, check=True)
else:
    print('Viewing the completed published results. Training is disabled.')


Viewing the completed published results. Training is disabled.


In [3]:
# Refresh to read completed results or the currently active reproduction.
result = PUBLISHED
active = BASE / 'active_run.json'
if RUN_TRAINING and active.exists():
    candidate = Path(json.loads(active.read_text())['directory'])
    if (candidate / 'summary.csv').exists():
        result = candidate
summary_path = result / 'summary.csv'
if summary_path.exists() and summary_path.stat().st_size > 1:
    summary = pd.read_csv(summary_path)
    display(summary[['model', 'completed_folds', 'complete', 'pr_auc', 'precision', 'recall', 'f1',
                     'pr_auc_sd', 'precision_sd', 'recall_sd', 'f1_sd']].round(4))
    print('PR-AUC is implemented as average precision (AP). Means and SDs are across matching folds.')
    print('Report:', result / 'comparison.html')
else:
    print('First fold is still training; no completed summary yet.')


                                   model  ...   f1_sd
0  LightGBM A2 (2k TF-IDF), fresh replay  ...  0.0975
1   Calibrated MLP + Extra Trees (50/50)  ...  0.1100

[2 rows x 11 columns]
PR-AUC is implemented as average precision (AP). Means and SDs are across matching folds.
Report: /private./reports/13_reference_comparison/comparison.html


In [4]:
paired_path = result / 'paired_differences.csv'
if paired_path.exists() and paired_path.stat().st_size > 1:
    display(pd.read_csv(paired_path).round(4))
else:
    print('Paired comparisons require at least two completed matching folds.')


              metric  matching_folds  ...    high  share_folds_a_better
0             pr_auc              25  ...  0.1883                  0.72
1          precision              25  ...  0.4061                  0.68
2             recall              25  ...  0.1405                  0.48
3                 f1              25  ...  0.1734                  0.56
4  precision_top5pct              25  ...  0.1532                  0.80
5     recall_top5pct              25  ...  0.1592                  0.80

[6 rows x 7 columns]


## Interpretation of the original completed run

Across all 25 matching folds, the calibrated MLP–Extra Trees ensemble achieved mean PR-AUC **0.6891**, versus **0.6142** for the fresh LightGBM replay. The ensemble had higher AP in **18/25 folds**, higher mean precision (0.7454 versus 0.6019) and F1 (0.5808 versus 0.5515), with slightly lower recall (0.5329 versus 0.5482). The approximate corrected AP-difference interval is **[-0.0385, 0.1883]**, so describe the observed advantage without claiming established superiority.

These are development results for two complete pipelines with different TF-IDF vocabulary limits. They do not establish that standalone MLP beats LightGBM. No reserved-test performance is reported. The original prediction files were checked against all 50 recorded outer-fold metric rows before publication. `completion.json` contains checksums of the published result files and the original run fingerprint; `manifest.json` preserves its original training provenance. Setup and platform support were subsequently packaged for teammates; estimator/calibration/evaluation implementations are unchanged.


## Live monitor for an optional reproduction

This cell refreshes every 10 seconds only when `RUN_TRAINING=True`. Interrupting it stops the display; the background worker continues. The worker lock determines whether training is active, so an old status file alone is not treated as evidence of a running process.

In [5]:
if RUN_TRAINING:
    import html, time
    from datetime import datetime
    from IPython.display import HTML, clear_output
    from src.reference13_runtime import worker_active
    while True:
        progress = {}
        active = BASE / 'active_run.json'
        if active.exists():
            directory = Path(json.loads(active.read_text())['directory'])
            progress_path = directory / 'progress.json'
            if progress_path.exists():
                progress = json.loads(progress_path.read_text())
        busy = worker_active(BASE)
        clear_output(wait=True)
        state = 'RUNNING' if busy else ('COMPLETE' if progress.get('complete') else 'NO ACTIVE WORKER')
        print(state, '| Refreshed:', datetime.now().astimezone().strftime('%Y-%m-%d %H:%M:%S %Z'))
        print('Completed folds:', progress.get('completed_folds', {}))
        log = BASE / 'run.log'
        if log.exists():
            display(HTML('<pre>' + html.escape('\n'.join(log.read_text(errors='replace').splitlines()[-15:])) + '</pre>'))
        if not busy:
            if not progress.get('complete'):
                print('Inspect the log; rerun the launch cell to resume checkpoints.')
            break
        time.sleep(10)
else:
    print('Original comparison complete: LightGBM 25/25; calibrated ensemble 25/25.')


Original comparison complete: LightGBM 25/25; calibrated ensemble 25/25.
